In [30]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader

In [55]:
script_name = source_path+"/scripts/torch_train_on_rep.py"
kind = 'body'
selected_FE = 'clip-vit-large-patch14'
data_augmentation = False
suffix = '_augmented' if data_augmentation else ''
train_filename,val_filename = file_IO.load_input_files(source_path,selected_FE,kind,suffix)
extra_view=False # works only with validation_mode=='val_only'
extra_integration_mode = 'concat'  # 'concat' or 'add'
aggregation_mode = None  # 'mean' or 'max'
if extra_view:
    extra_train_filename, extra_val_filename = file_IO.load_input_files(source_path,selected_FE,kind='body',suffix='')
else:
    extra_train_filename, extra_val_filename = None, None

selected_classifier='logreg'
validation_mode = 'val_only' #'kfold_train_only' #'val_only', #1fold_train_only
save_path = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\sklearn_model_trained_on_rep\\{selected_classifier}'
file_IO.access_or_create_dir(save_path)


In [56]:
#csv_location = 'icdar_EXTRACTED_train_df_clip-vit-large-patch14_20250517_144404.csv'
#parameters
script_name = source_path+"/scripts/sklearn_classifier_on_rep.py"
args = script_launching.DotDict(
    n_patches = -1,
    n_writers = -1,
    train_filename = train_filename,
    val_filename = val_filename,
    selected_model = selected_classifier,
    is_kaggle = False,
    with_pca = False,
    n_components = 0.95,
    validation_mode = validation_mode,  # 'kfold_train_only', 'val_only', '1fold_train_only'
    task = 'gender_detection',
    train_on_language = 'all',
    train_on_same = 'all',
    n_splits = 5,
    save_path = save_path,
    patch_merging = -1,
    data_augmentation = data_augmentation,
    extra_view=extra_view,
    extra_integration_mode = extra_integration_mode,
    extra_train_filename = extra_train_filename,
    extra_val_filename = extra_val_filename,
    aggregation_mode = aggregation_mode,
)
file_IO.save_args(args,save_path)  # Save the arguments to a file
script_launching.run_experiment_threaded(args,script_name)  # Test a single run first

Starting experiment:
[STDOUT] Running feature extraction script...
[STDOUT] aggregating patches, length before: 1128
[STDOUT] length after: 1128
[STDOUT] aggregating patches, length before: 284
[STDOUT] length after: 284
[STDOUT] Starting model cross-val...
[STDOUT] Fold 1 - IF Accuracy: 0.9725, IF Accuracy: 0.9725
[STDOUT] Fold 1 - OOF Accuracy: 0.6549, OOF Accuracy: 0.6549
[STDOUT] Average ensembled weighted accuracy: 0.6549
[STDOUT] Average individual accuracy: 0.6549
[STDOUT] Time taken to cross-validate the model: 1.37 seconds
[STDOUT] Model pipeline saved to file
Experiment finished with return code: 0


# reload

In [52]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()